# Mission 1: 신고자의 성별 분류 (음성 → gender)

Colab(GPU) 에서 위에서부터 순서대로 실행하세요. 실행 로그가 그대로 학습 로그로 제출됩니다.

In [ ]:
# ① GPU 확인 + Google Drive 연결 + 코드 받기
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/DCC'):
    !git clone -q -b ccr-1cdc76be-ibbw6s https://github.com/BioCode67/DCC.git /content/DCC
else:
    !git -C /content/DCC pull -q


In [ ]:
!pip -q install -r /content/DCC/mission1/requirements.txt

In [ ]:
# ② 데이터 폴더 지정 + 라벨(json)을 로컬 디스크로 병렬 복사
#   공유 폴더 '대학부 데이터' 를 내 드라이브에 '바로가기 추가' 해 둔 경로. 다르면 DATA_ROOT 만 고쳐주세요.
#   (Drive 는 파일 수가 많은 폴더를 읽는 게 느려서, 작은 json 은 32개씩 병렬로 로컬에 복사해 두고 사용.
#    wav 는 학습에 쓰는 것만 학습 중에 Drive 에서 바로 읽음)
import os, shutil, time
from concurrent.futures import ThreadPoolExecutor
DATA_ROOT = '/content/drive/MyDrive/2026 공모전/데이터+AI 크리에이터 캠프/대학부 데이터'
TRAIN_WAV  = f'{DATA_ROOT}/Training/1.원천데이터/TS_서울_구급'
TRAIN_JSON = f'{DATA_ROOT}/Training/2.라벨링데이터/TL_서울_구급'
VAL_WAV    = f'{DATA_ROOT}/Validation/1.원천데이터/VS_서울_구급'
VAL_JSON   = f'{DATA_ROOT}/Validation/2.라벨링데이터/VL_서울_구급'
for k, v in dict(TRAIN_WAV=TRAIN_WAV, TRAIN_JSON=TRAIN_JSON, VAL_WAV=VAL_WAV, VAL_JSON=VAL_JSON).items():
    print(f'{k:10s} {os.path.exists(v)}  {v}')

def local_copy(src, name, threads=32):
    # src 폴더의 파일들을 /content/data/name 으로 병렬 복사 (이미 있는 파일은 건너뜀)
    dst = f'/content/data/{name}'
    os.makedirs(dst, exist_ok=True)
    t0 = time.time()
    files = os.listdir(src)
    print(f'{name}: {len(files)} files 목록 읽기 {time.time()-t0:.0f}s')
    def cp(fn):
        d = os.path.join(dst, fn)
        if not os.path.exists(d):
            shutil.copyfile(os.path.join(src, fn), d)
    with ThreadPoolExecutor(threads) as ex:
        for i, _ in enumerate(ex.map(cp, files), 1):
            if i % 5000 == 0:
                print(f'  {i}/{len(files)}  ({time.time()-t0:.0f}s)')
    print(f'{name}: 복사 완료 {time.time()-t0:.0f}s')
    return dst

TRAIN_JSON = local_copy(TRAIN_JSON, 'TL')
VAL_JSON = local_copy(VAL_JSON, 'VL') if os.path.exists(VAL_JSON) else None


In [ ]:
# ③ 설정
#  - SAVE_DIR: 체크포인트/결과 저장 위치. 기본은 Colab 로컬 디스크(드라이브 용량 사용 안 함).
#              런타임이 끊기면 사라지므로 마지막 셀에서 꼭 다운로드할 것.
#              드라이브 여유가 1GB 이상이면 '/content/drive/MyDrive/DCC_outputs' 로 바꿔도 됨.
#  - QUICK_TEST: 처음엔 200 으로 전체 흐름만 확인 → 문제 없으면 0 으로 바꿔 본 학습
#  - LIMIT: 학습에 쓸 통화 수 (0=전체). 무료 Colab(T4, 시간 제한) 기준 권장값이 기본으로 들어가 있음.
SAVE_DIR = '/content/ckpt'
os.makedirs(SAVE_DIR, exist_ok=True)
QUICK_TEST = 200
LIMIT = 8000
N_LIMIT = QUICK_TEST or LIMIT
WORKERS = str(min(8, os.cpu_count()))


In [ ]:
# ④ 학습 (WavLM 특징 추출 → MLP 헤드 학습, Training 폴더만 사용)
import sys, time; sys.path.insert(0, '/content/DCC/mission1'); os.chdir('/content/DCC/mission1')
import train
args = train.get_parser().parse_args([
    '--train_audio_dir', TRAIN_WAV, '--train_label_dir', TRAIN_JSON,
    '--ckpt_path', f'{SAVE_DIR}/mission1.pt', '--cache_dir', '/content/cache',
    '--num_workers', WORKERS, '--limit', str(N_LIMIT)])
t0 = time.time(); train.main(args); print(f'총 학습 시간 {time.time()-t0:.0f}s')


In [ ]:
# ⑤ Validation 추론 (제출 명령어와 동일) + 채점 + 추론 시간 측정
!cd /content/DCC/mission1 && time python inference.py --audio_dir "$VAL_WAV" --label_dir "$VAL_JSON" \
    --ckpt_path "$SAVE_DIR/mission1.pt" --output ./outputs/mission1.csv --num_workers $WORKERS
!python /content/DCC/tools/evaluate.py --mission 1 --pred /content/DCC/mission1/outputs/mission1.csv --label_dir "$VAL_JSON"
!cp /content/DCC/mission1/outputs/mission1.csv "$SAVE_DIR/"


In [ ]:
# [마지막] 결과 다운로드 (런타임 종료 전에 꼭 실행!) — 체크포인트 + 결과 CSV 를 zip 으로 내려받기
import shutil
from google.colab import files
zip_path = shutil.make_archive('/content/' + os.path.basename(os.getcwd()) + '_outputs', 'zip', SAVE_DIR)
files.download(zip_path)
